# USMAN AND UMAR — OpenSwarm Colab Node

Run cells 1 to 6 in order. Each cell stops with a clear message if it fails.

Colab Secrets needed (key icon, enable notebook access):
- `UPSTASH_REDIS_REST_URL`, `UPSTASH_REDIS_REST_TOKEN`
- `APP_CLIENT_TOKEN` (must match the Android app)
- `OPENROUTER_API_KEY` or `NVIDIA_API_KEY`
- `TUNNEL_SUBDOMAIN` (optional)

Runtime: CPU is enough. If Colab shows "Failed to assign a backend", wait and retry.


In [ ]:
# 1. Clone the repository and create the virtual environment.
import os
os.chdir('/content')
!rm -rf /content/venv /content/usman-and-umar-app
!pip install -q virtualenv
!virtualenv -q /content/venv
!git clone -q --single-branch --branch feature/hybrid-task-orchestrator https://github.com/saedhawwa23-rgb/usman-and-umar-app.git /content/usman-and-umar-app
!git -C /content/usman-and-umar-app rev-parse HEAD
print("Repository cloned.")


In [ ]:
# 2. Install Python dependencies (foreground: any failure stops here).
!/content/venv/bin/pip install -q -r /content/usman-and-umar-app/cloud-backend/requirements.txt
!/content/venv/bin/pip install -q -r /content/usman-and-umar-app/cloud-backend/requirements-ai.txt
print("Python dependencies installed.")


In [ ]:
# 3. Playwright Chromium with Linux system libraries.
# Browsers are stored in a fixed path so install and the backend agree.
import os, subprocess

os.environ["PLAYWRIGHT_BROWSERS_PATH"] = "/content/ms-playwright"
VENV_BIN = "/content/venv/bin"
PW = f"{VENV_BIN}/playwright"
PY = f"{VENV_BIN}/python"

LIBS = [
    "libnss3", "libnspr4", "libatk1.0-0", "libatk-bridge2.0-0", "libcups2",
    "libdrm2", "libxkbcommon0", "libxcomposite1", "libxdamage1", "libxfixes3",
    "libxrandr2", "libgbm1", "libpango-1.0-0", "libcairo2", "libasound2",
    "libatspi2.0-0", "libxshmfence1", "fonts-liberation",
]

def run(cmd, label):
    print(f"\n== {label}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"{label} failed (exit {r.returncode}); see output above.")

run(["apt-get", "update", "-qq"], "apt update")
run(["apt-get", "install", "-y", "-qq", *LIBS], "system libraries")
run([PW, "install", "chromium"], "Chromium download")

exe = subprocess.run(
    [PY, "-c", "from playwright.sync_api import sync_playwright as s; p=s().start(); print(p.chromium.executable_path); p.stop()"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Chromium binary:", exe)

ldd = subprocess.run(["ldd", exe], capture_output=True, text=True).stdout
missing = [line for line in ldd.splitlines() if "not found" in line]
if missing:
    raise RuntimeError("Missing libraries:\n" + "\n".join(missing))

run([PY, "-c", (
    "from playwright.sync_api import sync_playwright as s\n"
    "p=s().start()\n"
    "b=p.chromium.launch(headless=True, args=['--no-sandbox','--disable-dev-shm-usage'])\n"
    "print('Chromium launch OK, version', b.version)\n"
    "b.close(); p.stop()\n"
)], "launch test")


In [ ]:
# 4. Load Colab Secrets (values are never printed) and check Upstash Redis.
import os, sys, uuid
from pathlib import Path
from google.colab import userdata

SECRET_NAMES = (
    "UPSTASH_REDIS_REST_URL", "UPSTASH_REDIS_REST_TOKEN", "APP_CLIENT_TOKEN",
    "OPENROUTER_API_KEY", "OPENROUTER_MODEL", "NVIDIA_API_KEY", "TUNNEL_SUBDOMAIN",
)
for name in SECRET_NAMES:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if isinstance(value, str) and value.strip():
        os.environ[name] = value.strip()
    else:
        os.environ.pop(name, None)

for required in ("UPSTASH_REDIS_REST_URL", "UPSTASH_REDIS_REST_TOKEN", "APP_CLIENT_TOKEN"):
    if not os.environ.get(required):
        raise RuntimeError(f"NOT READY: add {required} to Colab Secrets.")
if not (os.environ.get("OPENROUTER_API_KEY") or os.environ.get("NVIDIA_API_KEY")):
    raise RuntimeError("NOT READY: add OPENROUTER_API_KEY or NVIDIA_API_KEY to Colab Secrets.")

backend_dir = Path("/content/usman-and-umar-app/cloud-backend")
sys.path.insert(0, str(backend_dir))
from redis_task_storage import RedisTaskStorage
RedisTaskStorage().probe()

_nonce = uuid.uuid4().hex
os.environ["OPENSWARM_SERVER_ID"] = f"colab-{_nonce[:12]}"
os.environ["OPENSWARM_INSTANCE_ID"] = _nonce
print("Redis preflight passed. Secret values were not displayed.")


In [ ]:
# 5. Start FastAPI. The Playwright path set in cell 3 is inherited.
import os, subprocess
from pathlib import Path

backend_dir = Path('/content/usman-and-umar-app/cloud-backend')
os.chdir(backend_dir)
backend_log_path = Path('/content/fastapi.log')
backend_log = open(backend_log_path, 'w', buffering=1)
fastapi_process = subprocess.Popen(
    ['/content/venv/bin/python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'],
    cwd=str(backend_dir), stdout=backend_log, stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
print(f"FastAPI started (PID {fastapi_process.pid}); log: {backend_log_path}")


In [ ]:
# 6. Health gate. The tunnel starts only if every readiness flag is correct.
import requests, time, subprocess

health_url = 'http://127.0.0.1:7860/health'
required_true = ('ready', 'auth_ready', 'redis_ready', 'checkpoints_ready',
                 'checkpoint_storage_ready', 'lease_ready', 'lease_coordinator_ready', 'accepting_new_tasks')
required_false = ('draining',)
deadline = time.monotonic() + 180
health_ok, last_error = False, 'No health response yet.'

while time.monotonic() < deadline:
    if fastapi_process.poll() is not None:
        last_error = f"FastAPI exited with code {fastapi_process.returncode}."
        break
    try:
        r = requests.get(health_url, timeout=5)
        body = r.json() if r.headers.get('content-type', '').startswith('application/json') else None
        if r.status_code == 200 and isinstance(body, dict):
            if all(body.get(k) is True for k in required_true) and all(body.get(k) is False for k in required_false):
                health_ok = True
                break
            last_error = f"Not ready: {body}"
        else:
            last_error = f"HTTP {r.status_code}: {r.text[:300]}"
    except Exception as exc:
        last_error = f"{type(exc).__name__}: {exc}"
    time.sleep(3)

if not health_ok:
    if fastapi_process.poll() is None:
        fastapi_process.terminate()
    backend_log.flush()
    print('\n'.join(backend_log_path.read_text(errors='replace').splitlines()[-80:]))
    raise RuntimeError(f"NOT READY: tunnel not started. Last result: {last_error}")
print("Backend health: all readiness checks passed.")


In [ ]:
# 7. Public URL (Localtunnel). Starts only after cell 6 passes.
import os, re, subprocess, time
subdomain = os.environ.get("TUNNEL_SUBDOMAIN", "usman-umar-1").strip()
if not re.fullmatch(r"[a-zA-Z0-9-]{1,63}", subdomain):
    raise RuntimeError("TUNNEL_SUBDOMAIN has invalid characters.")
tunnel_log = open('/content/localtunnel.log', 'w')
tunnel_process = subprocess.Popen(
    ['npx', '--yes', 'localtunnel', '--port', '7860', '--subdomain', subdomain],
    stdout=tunnel_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)
time.sleep(5)
if tunnel_process.poll() is not None:
    raise RuntimeError(f"Localtunnel exited with code {tunnel_process.returncode}; see /content/localtunnel.log.")
print(f"Public URL: https://{subdomain}.loca.lt (open it once in Chrome to pass the reminder page)")


## Known limits

- Colab can stop the runtime without warning. Nothing drains in-flight tasks on shutdown.
- Redis writes are not atomic, so a passing health check does not prove exclusive task ownership.
- Localtunnel URLs are public. Keep `APP_CLIENT_TOKEN` secret.
- This notebook runs the OpenSwarm backend only.
